# 5 — Clasificador Jerárquico de Dos Etapas

En lugar de predecir directamente la temperatura exacta entre 20 clases, se divide el problema:

**Etapa 1 — Rango térmico** (3 clases)
- Frío: < 30 °C  (15, 18, 24)
- Templado: 30–55 °C  (30, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54)
- Caliente: > 55 °C  (56, 59, 61, 63, 68)

**Etapa 2 — Temperatura exacta** dentro del rango predicho
- Un sub-clasificador entrenado exclusivamente con muestras de ese rango

**Hipótesis:** separar primero el contexto físico (régimen frío/cálido) facilita la discriminación fina dentro de cada rango.

Se compara con el clasificador plano (RF / GB) como baseline.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

CSV_PATH  = Path("dataset_features_temperatura.csv")
FEAT_JSON = Path("features_seleccionadas.json")

In [ ]:
df = pd.read_csv(CSV_PATH)

META_COLS = {
    "sample_uid","sample_id","tx_path","rx_path",
    "tx_path_rel_to_bbdd_parent","rx_path_rel_to_bbdd_parent",
    "split","temperature_folder","N_subcarriers","M_symbols","num_valid","valid_ratio",
}
CONSTANT_COLS = {"phase_jump_m_count","phase_jump_m_ratio","phase_jump_k_count","phase_jump_k_ratio"}

if FEAT_JSON.exists():
    with open(FEAT_JSON) as f: FEAT_COLS = json.load(f)
else:
    FEAT_COLS = [c for c in df.columns if c not in META_COLS and c != "temperature" and c not in CONSTANT_COLS]
print(f"Features: {len(FEAT_COLS)}")

def range_label(t):
    if t < 30: return "frio"
    if t <= 55: return "templado"
    return "caliente"

df["range_label"] = df["temperature"].apply(range_label)
df["fine_label"]  = df["temperature"].astype(int).astype(str) + "C"

RANGES = ["frio", "templado", "caliente"]

def get_split(split):
    d = df[df["split"] == split]
    return d[FEAT_COLS].values, d["range_label"].values, d["fine_label"].values, d["temperature"].values

X_tr, y_tr_rng, y_tr_fine, t_tr = get_split("train")
X_v,  y_v_rng,  y_v_fine,  t_v  = get_split("val")
X_te, y_te_rng, y_te_fine, t_te = get_split("test")
print("Split OK")
print("Distribución rangos (train):", pd.Series(y_tr_rng).value_counts().to_dict())

In [ ]:
def make_rf():
    return Pipeline([("imp", SimpleImputer(strategy="mean")),
                     ("m", RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=42))])

# ── Etapa 1: clasificador de rango ─────────────────────────────────────────────
print("Etapa 1 — clasificador de rango (3 clases)...")
stage1 = make_rf()
stage1.fit(X_tr, y_tr_rng)
print(f"  Val acc: {accuracy_score(y_v_rng, stage1.predict(X_v))*100:.2f}%")
print(f"  Test acc: {accuracy_score(y_te_rng, stage1.predict(X_te))*100:.2f}%")

In [ ]:
# ── Etapa 2: sub-clasificadores por rango ──────────────────────────────────────
stage2 = {}
for rng in RANGES:
    mask_tr = y_tr_rng == rng
    X_sub = X_tr[mask_tr]; y_sub = y_tr_fine[mask_tr]
    n_classes = len(np.unique(y_sub))
    print(f"  Etapa 2 — {rng}: {X_sub.shape[0]} muestras, {n_classes} clases...", end=" ")
    m = make_rf()
    m.fit(X_sub, y_sub)
    # Evaluación interna (solo muestras del rango correcto en test)
    mask_te = y_te_rng == rng
    if mask_te.sum() > 0:
        acc2 = accuracy_score(y_te_fine[mask_te], m.predict(X_te[mask_te]))
        print(f"Test acc (solo rango correcto) = {acc2*100:.2f}%")
    else:
        print("sin muestras")
    stage2[rng] = m
print("Sub-clasificadores entrenados")

In [ ]:
# ── Predicción jerárquica ─────────────────────────────────────────────────────
def predict_hierarchical(X, stage1_model, stage2_models):
    preds_rng = stage1_model.predict(X)
    preds_fine = np.empty(len(X), dtype=object)
    for rng, model in stage2_models.items():
        mask = preds_rng == rng
        if mask.sum() > 0:
            preds_fine[mask] = model.predict(X[mask])
    return preds_rng, preds_fine

pred_rng_hier, pred_fine_hier = predict_hierarchical(X_te, stage1, stage2)
acc_hier = accuracy_score(y_te_fine, pred_fine_hier)
print(f"Jerárquico — Acc (20 clases) = {acc_hier*100:.2f}%")

# Baseline plano
baseline = make_rf()
baseline.fit(X_tr, y_tr_fine)
pred_flat = baseline.predict(X_te)
acc_flat = accuracy_score(y_te_fine, pred_flat)
print(f"Plano RF   — Acc (20 clases) = {acc_flat*100:.2f}%")

In [ ]:
# ── Accuracy por rango ────────────────────────────────────────────────────────
print("\nAcc por rango (jerárquico):")
for rng in RANGES:
    mask = y_te_rng == rng
    if mask.sum() > 0:
        acc_rng_hier = accuracy_score(y_te_fine[mask], pred_fine_hier[mask])
        acc_rng_flat = accuracy_score(y_te_fine[mask], pred_flat[mask])
        print(f"  {rng:10s}  Jerárquico={acc_rng_hier*100:.2f}%  Plano={acc_rng_flat*100:.2f}%")

In [ ]:
# ── Figura comparativa ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Acc global
ax = axes[0]
bars = ax.bar(["RF Plano", "Jerárquico"], [acc_flat*100, acc_hier*100],
              color=["#2E75B6", "#C00000"], alpha=0.85, width=0.4)
for b, v in zip(bars, [acc_flat*100, acc_hier*100]):
    ax.text(b.get_x()+b.get_width()/2, v+0.3, f"{v:.2f}%", ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("Accuracy [%]"); ax.set_ylim(0, 115)
ax.set_title("Accuracy global (20 clases)", fontsize=11, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

# Acc por rango
ax = axes[1]
xpos = np.arange(len(RANGES)); w = 0.35
acc_hier_rng = []
acc_flat_rng = []
for rng in RANGES:
    mask = y_te_rng == rng
    acc_hier_rng.append(accuracy_score(y_te_fine[mask], pred_fine_hier[mask])*100 if mask.sum() else 0)
    acc_flat_rng.append(accuracy_score(y_te_fine[mask], pred_flat[mask])*100 if mask.sum() else 0)
b1 = ax.bar(xpos-w/2, acc_flat_rng, w, label="RF Plano",   color="#2E75B6", alpha=0.85)
b2 = ax.bar(xpos+w/2, acc_hier_rng, w, label="Jerárquico", color="#C00000", alpha=0.85)
for b, v in list(zip(b1, acc_flat_rng)) + list(zip(b2, acc_hier_rng)):
    ax.text(b.get_x()+b.get_width()/2, v+0.3, f"{v:.1f}%", ha="center", fontsize=9)
ax.set_xticks(xpos); ax.set_xticklabels(RANGES, fontsize=10)
ax.set_ylabel("Accuracy [%]"); ax.set_ylim(0, 115)
ax.set_title("Accuracy por rango térmico", fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)

plt.suptitle("Clasificador Jerárquico vs RF Plano", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("jerarquico_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Matriz de confusión jerárquico vs plano ───────────────────────────────────
temps_sorted = sorted(df["fine_label"].unique())
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, (name, preds) in zip(axes, [("RF Plano", pred_flat),("Jerárquico", pred_fine_hier)]):
    cm = confusion_matrix(y_te_fine, preds, labels=temps_sorted)
    im = ax.imshow(cm, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(temps_sorted))); ax.set_xticklabels(temps_sorted, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(temps_sorted))); ax.set_yticklabels(temps_sorted, fontsize=8)
    for i in range(len(temps_sorted)):
        for j in range(len(temps_sorted)):
            if cm[i,j]>0:
                ax.text(j,i,cm[i,j],ha="center",va="center",fontsize=6,
                        color="white" if cm[i,j]>cm.max()*0.5 else "black")
    plt.colorbar(im, ax=ax, shrink=0.7)
    ax.set_title(f"{name}  Acc={accuracy_score(y_te_fine,preds)*100:.2f}%", fontsize=11, fontweight="bold")
plt.suptitle("Matrices de confusión — 20 clases", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("jerarquico_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Análisis de errores de etapa 1 (¿cuánto daña el error de rango?) ──────────
pred_rng_te = stage1.predict(X_te)
err_rng = pred_rng_te != y_te_rng
print(f"Muestras con rango mal predicho: {err_rng.sum()} / {len(err_rng)} ({err_rng.mean()*100:.2f}%)")

# ¿Cuántas muestras con rango correcto se clasifican bien en etapa 2?
mask_ok_rng = ~err_rng
acc_given_correct_rng = accuracy_score(y_te_fine[mask_ok_rng], pred_fine_hier[mask_ok_rng])
print(f"Acc etapa 2 dado rango correcto: {acc_given_correct_rng*100:.2f}%")
acc_given_wrong_rng = accuracy_score(y_te_fine[err_rng], pred_fine_hier[err_rng]) if err_rng.sum() > 0 else 0
print(f"Acc etapa 2 dado rango INCORRECTO: {acc_given_wrong_rng*100:.2f}%")